# Parking Slot Detection using YOLOv8

Notebook ini digunakan untuk melakukan training model deteksi slot parkir menggunakan YOLOv8.

## 0. Import

### a. Dependencies

In [1]:
!pip install ultralytics roboflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 144.1 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13


### b. Library

In [2]:
from roboflow import Roboflow
from ultralytics import YOLO
from google.colab import drive

import shutil
import torch
import os

os.makedirs("/content/drive/MyDrive/parking", exist_ok=True)

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
True
Tesla T4


### c. Datasets

In [3]:
rf = Roboflow(api_key="UVd5yq823wxqmLV8yl88")
project = rf.workspace("darrens-workspace-hcq4z").project("parking-slot-2-gegbi")
version = project.version(1)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Parking-Slot-[2]-1 in yolov8:: 100%|██████████| 1163/1163 [00:00<00:00, 4807.24it/s]


# 1. Train

In [4]:
model = YOLO("yolov8s-seg.pt")

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=15,
    save=True,
    plots=True,
)

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Parking-Slot-[2]-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, pati

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a2a20101dc0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

## 2. Evaluate

In [5]:
metrics = model.val(
    data=f"{dataset.location}/data.yaml",
    split="test"
)

print(f"mAP50: {metrics.box.map50}")
print(f"mAP50-95: {metrics.box.map}")
print(f"Precision: {metrics.box.mp}")
print(f"Recall: {metrics.box.mr}")

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8s-seg summary (fused): 86 layers, 11,779,987 parameters, 0 gradients, 39.9 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 17.4±8.5 MB/s, size: 41.5 KB)
val: Scanning /content/Parking-Slot-[2]-1/test/labels... 36 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 36/36 340.4it/s 0.1s
val: New cache created: /content/Parking-Slot-[2]-1/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.3it/s 2.3s
                   all         36         83      0.999          1      0.995      0.944      0.999          1      0.995      0.888
Speed: 8.6ms preprocess, 26.9ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to /content/runs/segment/val
mAP50: 0.995
mAP50-95: 0.9440873599848484
Precision: 0.9992585799058723
Recall: 1.0


In [6]:
model.predict(
    source=f"{dataset.location}/test/images",
    conf=0.5,
    save=True,
    show_labels=True,
    show_conf=True
)


image 1/36 /content/Parking-Slot-[2]-1/test/images/capture_20260524_000243_jpg.rf.bb33ccfcc54252c266ec07e13b8440e3.jpg: 640x640 (no detections), 20.8ms
image 2/36 /content/Parking-Slot-[2]-1/test/images/capture_20260524_000250_jpg.rf.69d4564d41483f5c74584e1ae9cd5210.jpg: 640x640 (no detections), 20.8ms
image 3/36 /content/Parking-Slot-[2]-1/test/images/capture_20260524_000743_jpg.rf.9dfd13492baae9ea14de5bb252a02084.jpg: 640x640 1 car, 20.7ms
image 4/36 /content/Parking-Slot-[2]-1/test/images/capture_20260524_000808_jpg.rf.5f6d07838ccfef75c636ab95ebee0155.jpg: 640x640 1 car, 19.9ms
image 5/36 /content/Parking-Slot-[2]-1/test/images/capture_20260524_001248_jpg.rf.8f54b71500146611dd84083c994be148.jpg: 640x640 2 cars, 19.8ms
image 6/36 /content/Parking-Slot-[2]-1/test/images/capture_20260524_001435_jpg.rf.61de0cb6aca174dc113d7f80cce208b4.jpg: 640x640 2 cars, 19.8ms
image 7/36 /content/Parking-Slot-[2]-1/test/images/capture_20260524_001450_jpg.rf.e37f2d83dbd751fba1c52bf9fcd41440.jpg: 640x6

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'car'}
 obb: None
 orig_img: array([[[ 46,  88,  81],
         [ 40,  82,  75],
         [ 35,  79,  72],
         ...,
         [213, 190, 158],
         [214, 191, 159],
         [215, 192, 160]],
 
        [[ 48,  90,  83],
         [ 41,  85,  78],
         [ 40,  84,  77],
         ...,
         [213, 190, 158],
         [214, 191, 159],
         [215, 192, 160]],
 
        [[ 48,  92,  85],
         [ 43,  90,  82],
         [ 42,  89,  81],
         ...,
         [213, 190, 158],
         [214, 191, 159],
         [214, 191, 159]],
 
        ...,
 
        [[253, 235, 212],
         [252, 234, 211],
         [251, 233, 210],
         ...,
         [ 33,  37,  31],
         [ 32,  36,  30],
         [ 32,  37,  28]],
 
        [[252, 235, 209],
         [252, 235, 209],
         [251, 234, 208],
         ...,
         [ 35,  37,  

In [8]:
shutil.copy(
    "runs/segment/train/weights/best.pt",
    "/content/drive/MyDrive/parking/best.pt"
)

'/content/drive/MyDrive/parking/best.pt'